# 00 — CFT Ingest & Flagging

Downloads the Colorado Forest Tracker statewide treatment polygons, cleans geometry,
and flags records where the GIS polygon is much larger than reported management acres
(possibly NEPA boundary or private parcel, etc).

**Outputs**
- `data/spatial/raw/CFTv2_CO_AllTreatments.gpkg` — raw REST download
- `data/spatial/mod/CFTv2_CO_Flagged.gpkg` — all records with flag columns added


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# --- Paths
code_dir = Path.cwd().parent          # .../treatment_interactions/code/
proj_dir = code_dir.parent            # .../treatment_interactions/
sys.path.insert(0, str(code_dir))

from cft_interactions.ingest import (
    load_or_download,
    clean_geometry,
    flag_acre_mismatch,
    MIN_AREA_ACRES,
)

RAW_FP     = proj_dir / 'data/spatial/raw/CFTv2_CO_AllTreatments.gpkg'
FLAGGED_FP = proj_dir / 'data/spatial/mod/CFTv2_CO_AllTreatments_Flagged.gpkg'
PROJ_CRS   = 26913   # NAD83 UTM Zone 13N

print(f'Project dir: {proj_dir}')
print(f'Min area threshold: {MIN_AREA_ACRES:.4f} acres')

## 01_Download or load

Either download a new copy of the CFT or load the existing source file.

In [ ]:
cft = load_or_download(RAW_FP, crs=PROJ_CRS)

print(f'\nN = {len(cft):,}  ({cft["YEAR_COMP"].min()}–{cft["YEAR_COMP"].max()})')
print(f'Agencies:       {len(cft["AGENCY_C"].unique())}')
print(f'Funding sources:{len(cft["FUND_SOURCE"].unique())}')
print(f'Activities:\n  {cft["ACTIVITY"].unique()}')
print(f'CRS: {cft.crs}')

## 02_Treatment heatmap

In [ ]:
from cft_interactions.plotting import plot_treatment_heatmap

# --- Load CO county boundaries as the reference layer
# Update this path to point to your local TIGER counties shapefile
COUNTIES_FP = Path.home() / 'Library/CloudStorage/Box-Box/MCC/data/boundaries/political/TIGER/tl_2024_us_county/tl_2024_us_county.shp'
counties = gpd.read_file(COUNTIES_FP).to_crs(PROJ_CRS)
co_counties = counties[counties['STATEFP'] == '08']  # Colorado FIPS

fig, ax = plot_treatment_heatmap(
    cft,
    boundary=co_counties,
    grid_size=5000,                 # 5 km cells
    title=f'Colorado Treatment Heatmap ({cft["YEAR_COMP"].min()}–{cft["YEAR_COMP"].max()})',
    out_fp=proj_dir / 'figures/CO_Treatment_Heatmap.png',
)
plt.show()

## 02_Geometry cleaning

In [ ]:
cft = clean_geometry(cft, snap_m=1.0)
print(f'After cleaning: {len(cft):,} features')
print(f'GIS acres distribution:\n{cft["gis_acres"].describe()}')

## 03a_Acre-mismatch flagging

Runs a log-log regression on GIS acres ~ MGT acres to identify erroneous polygons (management unit or NEPA boundary, for example).

In [ ]:
cft = flag_acre_mismatch(cft, n_std=3.0)
print(f'\nFlag summary:')
print(f'  ac_flag (residual): {cft["ac_flag"].sum():,}')
print(f'  ac_flag_abs:        {cft["ac_flag_abs"].sum():,}')
print(f'  Either flag:        {(cft["ac_flag"] | cft["ac_flag_abs"]).sum():,}')

## 03b_Diagnostic plots

Display the results from the log-log OLS regression highlighting which treatment units were flagged as erroneous.

In [ ]:
# --- Make a valid data mask (non-null / non-zero)
mask_valid = (cft['gis_acres'] > 0) & (cft['ACRES_MGT'] > 0)
df = cft[mask_valid].copy()

# --- Gather the model variables
df['log_gis'] = np.log10(df['gis_acres'])
df['log_mgt'] = np.log10(df['ACRES_MGT'])
retained = df[~df['ac_flag']]
flagged  = df[df['ac_flag']]

# --- 4-panel plot
fig, axes = plt.subplots(2, 2, figsize=(8, 6))

# Log-log scatter
ax = axes[0, 0]
ax.scatter(retained['log_gis'], retained['log_mgt'], s=3, alpha=0.3,
           color='steelblue', label='Retained', rasterized=True)
ax.scatter(flagged['log_gis'], flagged['log_mgt'], s=10, alpha=0.7,
           color='darkorange', label='Flagged', zorder=3)
x_r = np.linspace(df['log_gis'].min(), df['log_gis'].max(), 100)
sl, ic, *_ = stats.linregress(df['log_gis'], df['log_mgt'])
ax.plot(x_r, ic + sl * x_r, color='gray', lw=1.2, ls='--', label='OLS')
ax.set_xlabel('log10(ACRES_GIS)'); ax.set_ylabel('log10(ACRES_MGT)')
ax.set_title('Log-log scatter', fontsize=10); ax.legend(fontsize=8)

# Residual distribution
ax = axes[0, 1]
resid_std = df['log_ac_resid'].std()
threshold = -3 * resid_std
bins = np.linspace(df['log_ac_resid'].min(), df['log_ac_resid'].max(), 50)
ax.hist(retained['log_ac_resid'], bins=bins, color='steelblue', alpha=0.7, label='Retained')
ax.hist(flagged['log_ac_resid'], bins=bins, color='darkorange', alpha=0.8, label='Flagged')
ax.axvline(threshold, color='crimson', ls='--', lw=1.5, label=f'−3σ = {threshold:.2f}')
ax.set_xlabel('Residual (log10)'); ax.set_ylabel('Count')
ax.set_title('Residual distribution', fontsize=10); ax.legend(fontsize=8)

# Flagged by activity — bar length = median acres_diff, label shows n + median
ax = axes[1, 0]
act_stats = (flagged.groupby('ACTIVITY')['acres_diff']
             .median().sort_values(ascending=True))
ax.barh(act_stats.index, act_stats.values, color='darkorange', alpha=0.8)
for i, (act, val) in enumerate(act_stats.items()):
    n = (flagged['ACTIVITY'] == act).sum()
    ax.text(val + 15, i, f'n={n}, med={val:.0f} ac', va='center', fontsize=7, color='dimgray')
ax.set_xlim(0, act_stats.max() * 1.5)
ax.set_xlabel('Median acres_diff (flagged records)')
ax.set_title('Flagged by activity', fontsize=10)
ax.tick_params(axis='y', labelsize=8)

# Flag rate by size bin
ax = axes[1, 1]
bin_edges  = [0, 1, 10, 100, 1000, 10000, np.inf]
bin_labels = ['<1', '1–10', '10–100', '100–1k', '1k–10k', '>10k']
df['size_bin'] = pd.cut(df['ACRES_GIS'], bins=bin_edges, labels=bin_labels)
flag_rates = df.groupby('size_bin', observed=True)['ac_flag'].mean() * 100
ax.bar(flag_rates.index, flag_rates.values, color='steelblue', alpha=0.8)
ax.axhline(df['ac_flag'].mean() * 100, color='gray', ls='--', lw=1.2)
ax.set_xlabel('ACRES_GIS bin'); ax.set_ylabel('Flag rate (%)')
ax.set_title('Flag rate by size class', fontsize=10)

plt.tight_layout()
fig_fp = proj_dir / 'figures/LogLog_Acres_Diagnostics.png'
fig_fp.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_fp, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Save the flagged file out
FLAGGED_FP.parent.mkdir(parents=True, exist_ok=True)
cft.to_file(FLAGGED_FP)
print(f'Saved {len(cft):,} features → {FLAGGED_FP}')
print(f'Columns: {list(cft.columns)}')